# P07 — Prédiction de l'indice de qualité de l'air ATMO

**Contexte métier.** AASQA (associations agréées de surveillance de la qualité de l'air), Agences régionales de santé, applications grand public : anticiper les pics de pollution pour déclencher les mesures d'urgence (circulation différenciée, gratuité des transports) et alerter les personnes sensibles. Enjeu : 48 000 décès/an attribuables à la pollution de l'air extérieur (Santé publique France).

**Objectif de l'EDA.** Identifier la cible (indice ATMO 0-10) et ses relations avec les sous-indices de polluants, la saison et le jour.

**Données.** Indice ATMO Pays de la Loire 2017-2020 — Air Pays de la Loire via data.paysdelaloire.fr. Licence ouverte Etalab 2.0. 2 562 lignes = 7 agglomérations × 366 jours de l'année 2020.

## 1. Téléchargement

Export CSV du portail régional :

In [ ]:
import urllib.request, os
os.makedirs("data", exist_ok=True)

URL = "https://data.paysdelaloire.fr/api/explore/v2.1/catalog/datasets/qualite-de-lair-indice-atmo-pays-de-la-loire-2017-2020/exports/csv"
if not os.path.exists("data/indice_atmo_pdl_2017_2020.csv"):
    urllib.request.urlretrieve(URL, "data/indice_atmo_pdl_2017_2020.csv")
print(os.path.getsize("data/indice_atmo_pdl_2017_2020.csv") / 1e6, "Mo")

## 2. Chargement

Séparateur `;`. L'indice ATMO (0 très bon → 10 très mauvais) est fourni avec ses sous-indices par polluant, la qualification, la zone et la date.

In [ ]:
import pandas as pd, numpy as np

df = pd.read_csv("data/indice_atmo_pdl_2017_2020.csv", sep=";",
                 usecols=lambda c: c not in ("geom", "Geo Point"))
df.columns = [c.strip() for c in df.columns]
df["Date"] = pd.to_datetime(df["Date"])
df["mois"] = df["Date"].dt.month
df["dow"] = df["Date"].dt.dayofweek
print(len(df), "lignes,", df["Zone"].nunique(), "zones")

## 3. Analyse

Distribution de l'indice, saisonnalité mensuelle, corrélations entre l'indice ATMO et ses sous-indices (PM10, PM2.5, O3, NO2, SO2), et jours dégradés par zone.

In [ ]:
print("Distribution indice ATMO :", df["Indice ATMO"].value_counts().sort_index().to_dict())
print("Qualification :", df["qualif"].value_counts().to_dict())
print()
print("Indice moyen par mois :", df.groupby("mois")["Indice ATMO"].mean().round(2).to_dict())
print("Indice moyen par zone :", df.groupby("Zone")["Indice ATMO"].mean().round(2).to_dict())
print("Jours mauvais (>=8) par zone :")
print((df.assign(mauvais=df["Indice ATMO"] >= 8).groupby("Zone")["mauvais"].mean() * 100).round(1).to_dict())
sub = ["Indice particules fines pm10", "Indice particules fines pm2.5",
       "Indice ozone (O3)", "Indice dioxyde d'azote (no2)", "Indice dioxyde de soufre (SO2)"]
print("Corrélations Spearman sous-indices vs ATMO :")
print(df[["Indice ATMO"] + sub + ["mois", "dow"]].corr("spearman")["Indice ATMO"].round(2).to_dict())

## 4. Résultats observés (exécution du 16/10/2026)

- n lignes : 2562
- periode : 2019-12-31 → 2020-12-30
- zones : Agglomération du Choletais : 366, Angers Loire Métropole : 366, La Roche-sur-Yon Agglomération : 366, Laval Aggloméra…
- indice atmo dist : 0 : 7, 2 : 66, 3 : 1199, 4 : 841, 5 : 332, 6 : 97, 7 : 14, 8 : 6
- qualif dist : bon : 2040, moyen : 332, médiocre : 111, très bon : 66, nan : 7, mauvais : 6
- indice moyen par zone : Agglomération du Choletais : 3.75, Angers Loire Métropole : 3.67, La Roche-sur-Yon Agglomération : 3.67, Laval Agglom…
- indice moyen par mois : 1 : 3.58, 2 : 3.76, 3 : 3.98, 4 : 4.35, 5 : 4.32, 6 : 3.70, 7 : 3.63, 8 : 3.79, 9 : 3.78, 10 : 2.97
- indice moyen par jour : 0 : 3.68, 1 : 3.75, 2 : 3.74, 3 : 3.83, 4 : 3.72, 5 : 3.49, 6 : 3.71
- pct jours mauvais par zone : Agglomération du Choletais : 0.30, Angers Loire Métropole : 0.30, La Roche-sur-Yon Agglomération : 0.30, Laval Agglom…
- corr spearman sous indices : Indice ATMO : 1.00, Indice particules fines pm10 : 0.46, Indice particules fines pm2.5 : nan, Indice ozone (O3) : 0.6…
- missing pct : code zone : 0.00, Zone : 0.00, Type Zone : 0.00, Date : 0.00, Indice ATMO : 0.00, type : 0.00, Indice particules fine…

## 5. Variable cible et régresseurs

**Cible (classification ordinale) : `Indice ATMO` (0-10)**, ou sa version binaire métier " journée dégradée " (indice ≥ 8) pour les alertes.

**Régresseurs retenus (côté variables météo/temps si l'on veut un modèle prédictif) :**
- `mois` : maximum en mars-avril (3,8) et hiver ; minimum estival — saisonnalité nette
- `Zone` : Nantes/Saint-Nazaire légèrement plus dégradées que Laval
- Pour un modèle explicatif immédiat : les sous-indices `pm10`, `pm2.5`, `O3`, `no2` (corr. 0,5-0,7 avec l'ATMO, l'indice = max des sous-indices)

**Pièges :**
- L'indice ATMO est par construction le maximum des sous-indices : un modèle appris sur les sous-indices du jour ne "prédit" rien — il faut soit prédire les polluants puis composer, soit modéliser directement J+1 à partir de J et de la météo prévue
- Le jeu couvre une seule année (2020, confinement !) : généralisation limitée, à corroborer avec les données 2021+ (portail Atmo France via API WFS)
- Pas de variable météo incluse : la joindre (température, vent) est le vrai levier prédictif
- 7 lignes avec indice manquant : gérer les NaN